# Hierarchical Multi-Granularity Sentiment Analysis for E-Commerce Reviews
## MSc Research Notebook

This notebook is fully self-contained and runs on:

- Apple Silicon Mac via MPS
- NVIDIA GPU via CUDA
- CPU as a fallback
- Google Colab (CPU or GPU)

It performs the following steps end-to-end:

1. Auto-detects the device and installs missing dependencies
2. Streams real Amazon Reviews 2023 data from HuggingFace (Electronics subset)
3. Maps 1-5 star ratings to 3-class sentiment with class balancing
4. Trains three models on the same data:
   - TF-IDF plus Logistic Regression baseline
   - BERT-BiLSTM-Attention
   - HMGS hierarchical document model with sentence-level attention pooling
5. Reports accuracy, macro-F1, classification report and confusion matrices
6. Saves every plot to `./artifacts/figures/` and every checkpoint to `./artifacts/models/`
7. Provides an interactive inference function so any user-supplied review is
   passed through all three models with a probability bar chart and a
   sentence-attention plot


## 1. Install Dependencies

In [ ]:
# Install missing packages. Safe to re-run; only installs what is missing.
import importlib
import subprocess
import sys


def ensure(spec, import_name):
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])


for spec, mod in [
    ("torch>=2.1", "torch"),
    ("transformers>=4.40", "transformers"),
    ("datasets>=2.18", "datasets"),
    ("scikit-learn>=1.3", "sklearn"),
    ("matplotlib>=3.8", "matplotlib"),
    ("seaborn>=0.13", "seaborn"),
    ("nltk>=3.8", "nltk"),
    ("tqdm>=4.66", "tqdm"),
    ("pandas>=2.0", "pandas"),
]:
    ensure(spec, mod)
print("Dependencies ready.")


## 2. Device Detection and Imports

In [ ]:
import gc
import json
import os
import pickle
import platform
import random
import sys
import time

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (BertModel, BertTokenizerFast,
                          get_linear_schedule_with_warmup)

IN_COLAB = "google.colab" in sys.modules
HAS_CUDA = torch.cuda.is_available()
HAS_MPS = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

if HAS_CUDA:
    DEVICE = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif HAS_MPS:
    DEVICE = torch.device("mps")
    print("Using Apple Silicon MPS")
else:
    DEVICE = torch.device("cpu")
    print(f"Using CPU on {platform.platform()}")

print(f"PyTorch:   {torch.__version__}")
print(f"In Colab:  {IN_COLAB}")

for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.data.find(f"tokenizers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)


## 3. Configuration

In [ ]:
ARTIFACTS = "./artifacts"
FIGURES = os.path.join(ARTIFACTS, "figures")
MODELS = os.path.join(ARTIFACTS, "models")
os.makedirs(FIGURES, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

SEED = 42

# Hyperparameters chosen so the notebook completes in a reasonable time
# even on CPU. Increase TRAIN_SAMPLES and EPOCHS on a real GPU for stronger
# numbers in the thesis report.
MAX_LEN       = 96      # BERT tokens per sentence
MAX_SENTS     = 6       # max sentences per review for HMGS
TRAIN_SAMPLES = 3000    # balanced 3-way
VAL_SAMPLES   = 600
TEST_SAMPLES  = 600
BATCH_SIZE    = 8
EPOCHS        = 3
LR_BERT       = 2e-5
LR_HEAD       = 1e-3
WEIGHT_DECAY  = 0.01

HF_DATASET = "McAuley-Lab/Amazon-Reviews-2023"
# Falls back to All_Beauty if the requested category is unavailable
HF_CONFIG_PRIMARY = "raw_review_All_Electronics"
HF_CONFIG_FALLBACK = "raw_review_All_Beauty"

LABELS = ["Negative", "Neutral", "Positive"]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if HAS_CUDA:
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (8, 5), "savefig.dpi": 120, "font.size": 10})
print(f"Artifacts will be written under: {os.path.abspath(ARTIFACTS)}")


## 4. Data Loading

Streams Amazon Reviews 2023 from HuggingFace, balances classes by sampling an
equal number of Negative (1-2 stars), Neutral (3 stars) and Positive (4-5
stars) reviews. Streaming avoids downloading tens of millions of records.

In [ ]:
def stars_to_label(rating):
    r = float(rating)
    if r <= 2:
        return 0
    if r >= 4:
        return 2
    return 1


def fetch_balanced(config_name, total_per_class):
    print(f"Streaming {HF_DATASET} / {config_name} ...")
    ds = load_dataset(
        HF_DATASET, config_name, split="full",
        streaming=True, trust_remote_code=True,
    )
    buckets = {0: [], 1: [], 2: []}
    seen = 0
    for row in ds:
        seen += 1
        text = ((row.get("title") or "") + ". " + (row.get("text") or "")).strip(". ").strip()
        if len(text) < 15:
            continue
        rating = row.get("rating")
        if rating is None:
            continue
        lab = stars_to_label(rating)
        if len(buckets[lab]) < total_per_class:
            buckets[lab].append(text)
        if all(len(v) >= total_per_class for v in buckets.values()):
            break
        if seen >= total_per_class * 100:
            break
    print(f"Collected: Neg={len(buckets[0])} Neu={len(buckets[1])} Pos={len(buckets[2])} (scanned {seen})")
    return buckets


need = (TRAIN_SAMPLES + VAL_SAMPLES + TEST_SAMPLES) // 3 + 50
HF_CONFIG = HF_CONFIG_PRIMARY
try:
    buckets = fetch_balanced(HF_CONFIG, need)
except Exception as exc:
    print(f"{HF_CONFIG_PRIMARY} unavailable ({exc}); falling back to {HF_CONFIG_FALLBACK}")
    HF_CONFIG = HF_CONFIG_FALLBACK
    buckets = fetch_balanced(HF_CONFIG, need)

texts, labels = [], []
for c, items in buckets.items():
    texts.extend(items)
    labels.extend([c] * len(items))

paired = list(zip(texts, labels))
random.Random(SEED).shuffle(paired)
texts = [t for t, _ in paired]
labels = [l for _, l in paired]
print(f"Total reviews collected: {len(texts)}")


## 5. Train / Validation / Test Split

In [ ]:
assert len(texts) >= TRAIN_SAMPLES + VAL_SAMPLES + TEST_SAMPLES, (
    f"Not enough data ({len(texts)})."
)
texts_tr = texts[:TRAIN_SAMPLES]
lbl_tr = labels[:TRAIN_SAMPLES]
texts_va = texts[TRAIN_SAMPLES:TRAIN_SAMPLES + VAL_SAMPLES]
lbl_va = labels[TRAIN_SAMPLES:TRAIN_SAMPLES + VAL_SAMPLES]
texts_te = texts[TRAIN_SAMPLES + VAL_SAMPLES:TRAIN_SAMPLES + VAL_SAMPLES + TEST_SAMPLES]
lbl_te = labels[TRAIN_SAMPLES + VAL_SAMPLES:TRAIN_SAMPLES + VAL_SAMPLES + TEST_SAMPLES]
print(f"Train={len(texts_tr)}  Val={len(texts_va)}  Test={len(texts_te)}")
print(f"Train class counts: {np.bincount(lbl_tr).tolist()}")
print(f"Val   class counts: {np.bincount(lbl_va).tolist()}")
print(f"Test  class counts: {np.bincount(lbl_te).tolist()}")


In [ ]:
# Data overview figure: class distribution and review length
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = np.bincount(labels, minlength=3)
axes[0].bar(LABELS, counts, color=["#ef4444", "#f59e0b", "#10b981"])
axes[0].set_title(f"Class Distribution ({HF_CONFIG})")
axes[0].set_ylabel("Count")
for i, c in enumerate(counts):
    axes[0].text(i, c + max(counts) * 0.01, str(c), ha="center")

lengths = [len(t.split()) for t in texts]
axes[1].hist(lengths, bins=40, color="#6366f1", edgecolor="white")
axes[1].set_title("Review Length (words)")
axes[1].set_xlabel("Words")
axes[1].set_ylabel("Frequency")
axes[1].axvline(np.mean(lengths), color="red", linestyle="--",
                label=f"mean={np.mean(lengths):.0f}")
axes[1].legend()

fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "01_data_overview.png"))
plt.show()


## 6. Model Definitions

In [ ]:
class BiLSTMAttention(nn.Module):
    """BERT -> Bidirectional LSTM -> additive attention -> classifier."""

    def __init__(self, bert, lstm_h=256, num_labels=3, dropout=0.3):
        super().__init__()
        self.bert = bert
        h = bert.config.hidden_size
        self.lstm = nn.LSTM(h, lstm_h, batch_first=True, bidirectional=True)
        self.attn_w = nn.Linear(lstm_h * 2, 1)
        self.clf = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_h * 2, lstm_h),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_h, num_labels),
        )

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        lstm_out, _ = self.lstm(out)
        scores = self.attn_w(lstm_out).squeeze(-1)
        scores = scores.masked_fill(~attention_mask.bool(), float("-inf"))
        weights = F.softmax(scores, dim=-1)
        ctx = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)
        logits = self.clf(ctx)
        result = {"logits": logits, "attention": weights}
        if labels is not None:
            result["loss"] = F.cross_entropy(logits, labels)
        return result


class HMGS(nn.Module):
    """Hierarchical document model.

    Splits a review into sentences, embeds each sentence with the shared
    BERT encoder, and aggregates the per-sentence CLS representations using a
    learnable query-based attention layer to produce a document representation
    used for 3-class sentiment classification.
    """

    def __init__(self, bert, num_labels=3, dropout=0.3):
        super().__init__()
        self.bert = bert
        h = bert.config.hidden_size
        self.query = nn.Parameter(torch.randn(h))
        self.key_proj = nn.Linear(h, h)
        self.scale = h ** 0.5
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(h, h // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(h // 2, num_labels),
        )

    def encode_sentences(self, ids, mask):
        B, S, L = ids.shape
        flat_ids = ids.view(B * S, L)
        flat_mask = mask.view(B * S, L)
        is_real = flat_mask.sum(-1) > 0
        h = self.bert.config.hidden_size
        cls = torch.zeros(B * S, h, device=ids.device)
        if is_real.any():
            out = self.bert(input_ids=flat_ids[is_real],
                            attention_mask=flat_mask[is_real])
            cls[is_real] = out.last_hidden_state[:, 0, :]
        return cls.view(B, S, h)

    def forward(self, ids, mask, sent_mask, labels=None):
        sent_reprs = self.encode_sentences(ids, mask)
        keys = self.key_proj(sent_reprs)
        scores = torch.einsum("bsh,h->bs", keys, self.query) / self.scale
        scores = scores.masked_fill(~sent_mask.bool(), float("-inf"))
        attn = F.softmax(scores, dim=-1)
        doc = torch.einsum("bs,bsh->bh", attn, sent_reprs)
        logits = self.classifier(doc)
        result = {"logits": logits, "attention": attn}
        if labels is not None:
            result["loss"] = F.cross_entropy(logits, labels)
        return result


## 7. Tokenizer, Datasets and Training Helpers

In [ ]:
class FlatDataset(Dataset):
    """Whole-review tokenisation for BiLSTM and baseline."""

    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN * 2):
        self.texts = texts
        self.labels = labels
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], max_length=self.max_len,
                       padding="max_length", truncation=True, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[i], dtype=torch.long),
        }


class SentenceDataset(Dataset):
    """Splits each review into sentences for HMGS."""

    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN, max_sents=MAX_SENTS):
        self.texts = texts
        self.labels = labels
        self.tok = tokenizer
        self.max_len = max_len
        self.max_sents = max_sents

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        sents = nltk.sent_tokenize(self.texts[i])[:self.max_sents]
        if not sents:
            sents = [self.texts[i][:200]]
        n = len(sents)
        encs = [self.tok(s, max_length=self.max_len, padding="max_length",
                         truncation=True, return_tensors="pt") for s in sents]
        ids = [e["input_ids"].squeeze(0) for e in encs]
        masks = [e["attention_mask"].squeeze(0) for e in encs]
        while len(ids) < self.max_sents:
            ids.append(torch.zeros(self.max_len, dtype=torch.long))
            masks.append(torch.zeros(self.max_len, dtype=torch.long))
        sent_mask = torch.zeros(self.max_sents, dtype=torch.long)
        sent_mask[:n] = 1
        return {
            "input_ids": torch.stack(ids, 0),
            "attention_mask": torch.stack(masks, 0),
            "sent_mask": sent_mask,
            "labels": torch.tensor(self.labels[i], dtype=torch.long),
        }


def make_optimizer(model, lr_bert=LR_BERT, lr_head=LR_HEAD, wd=WEIGHT_DECAY):
    return torch.optim.AdamW([
        {"params": model.bert.parameters(), "lr": lr_bert},
        {"params": [p for n, p in model.named_parameters() if not n.startswith("bert.")],
         "lr": lr_head},
    ], weight_decay=wd)


def train_one_epoch(model, loader, opt, sched, device, hmgs=False):
    model.train()
    total = 0.0
    for batch in tqdm(loader, leave=False, desc="train"):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lab = batch["labels"].to(device)
        if hmgs:
            sm = batch["sent_mask"].to(device)
            r = model(ids, mask, sm, lab)
        else:
            r = model(ids, mask, lab)
        r["loss"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        total += r["loss"].item()
    return total / max(1, len(loader))


@torch.no_grad()
def evaluate(model, loader, device, hmgs=False):
    model.eval()
    preds, true = [], []
    for batch in tqdm(loader, leave=False, desc="eval"):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lab = batch["labels"].to(device)
        if hmgs:
            sm = batch["sent_mask"].to(device)
            r = model(ids, mask, sm)
        else:
            r = model(ids, mask)
        preds.extend(r["logits"].argmax(-1).cpu().numpy().tolist())
        true.extend(lab.cpu().numpy().tolist())
    return {
        "accuracy": float(accuracy_score(true, preds)),
        "macro_f1": float(f1_score(true, preds, average="macro")),
        "predictions": preds,
        "labels": true,
    }


tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
print("Tokenizer loaded.")


## 8. Train BERT-BiLSTM-Attention

In [ ]:
print("Training BERT-BiLSTM-Attention...")
set_seed(SEED)
bert_a = BertModel.from_pretrained("bert-base-uncased")
bilstm = BiLSTMAttention(bert_a).to(DEVICE)

ds_tr = FlatDataset(texts_tr, lbl_tr, tokenizer)
ds_va = FlatDataset(texts_va, lbl_va, tokenizer)
ds_te = FlatDataset(texts_te, lbl_te, tokenizer)
dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True)
dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE)
dl_te = DataLoader(ds_te, batch_size=BATCH_SIZE)

opt = make_optimizer(bilstm)
total_steps = len(dl_tr) * EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(total_steps * 0.1), total_steps)

bilstm_history = []
best_f1 = 0.0
for ep in range(EPOCHS):
    t0 = time.time()
    train_loss = train_one_epoch(bilstm, dl_tr, opt, sched, DEVICE, hmgs=False)
    val = evaluate(bilstm, dl_va, DEVICE, hmgs=False)
    bilstm_history.append({
        "epoch": ep + 1, "train_loss": train_loss,
        "val_accuracy": val["accuracy"], "val_macro_f1": val["macro_f1"],
    })
    print(f"Epoch {ep+1}/{EPOCHS} loss={train_loss:.4f} "
          f"val_acc={val['accuracy']:.4f} val_f1={val['macro_f1']:.4f} "
          f"({time.time()-t0:.0f}s)")
    if val["macro_f1"] > best_f1:
        best_f1 = val["macro_f1"]
        torch.save(bilstm.state_dict(), os.path.join(MODELS, "bilstm_best.pt"))
print(f"BiLSTM best val macro-F1: {best_f1:.4f}")


In [ ]:
# BiLSTM training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
xs = [h["epoch"] for h in bilstm_history]

axes[0].plot(xs, [h["train_loss"] for h in bilstm_history], "o-", color="#6366f1")
axes[0].set_title("BiLSTM Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")

axes[1].plot(xs, [h["val_accuracy"] for h in bilstm_history], "o-", label="Val Accuracy", color="#10b981")
axes[1].plot(xs, [h["val_macro_f1"] for h in bilstm_history], "s-", label="Val Macro F1", color="#f59e0b")
axes[1].set_title("BiLSTM Validation")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1); axes[1].legend()

fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "02_bilstm_training_curves.png"))
plt.show()


## 9. Train HMGS

In [ ]:
print("Training HMGS hierarchical document model...")
set_seed(SEED)
bert_b = BertModel.from_pretrained("bert-base-uncased")
hmgs = HMGS(bert_b).to(DEVICE)

# HMGS pushes (B * MAX_SENTS) sentences through BERT per batch, so use a
# smaller batch size to stay within memory limits on CPU and MPS.
hmgs_bs = max(BATCH_SIZE // 2, 2)

ds_h_tr = SentenceDataset(texts_tr, lbl_tr, tokenizer)
ds_h_va = SentenceDataset(texts_va, lbl_va, tokenizer)
ds_h_te = SentenceDataset(texts_te, lbl_te, tokenizer)
dl_h_tr = DataLoader(ds_h_tr, batch_size=hmgs_bs, shuffle=True)
dl_h_va = DataLoader(ds_h_va, batch_size=hmgs_bs)
dl_h_te = DataLoader(ds_h_te, batch_size=hmgs_bs)

opt_h = make_optimizer(hmgs)
total_steps_h = len(dl_h_tr) * EPOCHS
sched_h = get_linear_schedule_with_warmup(opt_h, int(total_steps_h * 0.1), total_steps_h)

hmgs_history = []
best_f1 = 0.0
for ep in range(EPOCHS):
    t0 = time.time()
    train_loss = train_one_epoch(hmgs, dl_h_tr, opt_h, sched_h, DEVICE, hmgs=True)
    val = evaluate(hmgs, dl_h_va, DEVICE, hmgs=True)
    hmgs_history.append({
        "epoch": ep + 1, "train_loss": train_loss,
        "val_accuracy": val["accuracy"], "val_macro_f1": val["macro_f1"],
    })
    print(f"Epoch {ep+1}/{EPOCHS} loss={train_loss:.4f} "
          f"val_acc={val['accuracy']:.4f} val_f1={val['macro_f1']:.4f} "
          f"({time.time()-t0:.0f}s)")
    if val["macro_f1"] > best_f1:
        best_f1 = val["macro_f1"]
        torch.save(hmgs.state_dict(), os.path.join(MODELS, "hmgs_best.pt"))
print(f"HMGS best val macro-F1: {best_f1:.4f}")


In [ ]:
# HMGS training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
xs = [h["epoch"] for h in hmgs_history]

axes[0].plot(xs, [h["train_loss"] for h in hmgs_history], "o-", color="#6366f1")
axes[0].set_title("HMGS Training Loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")

axes[1].plot(xs, [h["val_accuracy"] for h in hmgs_history], "o-", label="Val Accuracy", color="#10b981")
axes[1].plot(xs, [h["val_macro_f1"] for h in hmgs_history], "s-", label="Val Macro F1", color="#f59e0b")
axes[1].set_title("HMGS Validation")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_ylim(0, 1); axes[1].legend()

fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "03_hmgs_training_curves.png"))
plt.show()


## 10. TF-IDF + Logistic Regression Baseline

In [ ]:
print("Training TF-IDF + Logistic Regression baseline...")
vec = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, min_df=2)
X_tr = vec.fit_transform(texts_tr)
X_va = vec.transform(texts_va)
X_te = vec.transform(texts_te)

clf = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0)
clf.fit(X_tr, lbl_tr)

baseline_val_acc = float(accuracy_score(lbl_va, clf.predict(X_va)))
baseline_val_f1 = float(f1_score(lbl_va, clf.predict(X_va), average="macro"))
baseline_test_preds = clf.predict(X_te).tolist()
baseline_test_acc = float(accuracy_score(lbl_te, baseline_test_preds))
baseline_test_f1 = float(f1_score(lbl_te, baseline_test_preds, average="macro"))

print(f"Baseline val:  acc={baseline_val_acc:.4f}  macro-F1={baseline_val_f1:.4f}")
print(f"Baseline test: acc={baseline_test_acc:.4f}  macro-F1={baseline_test_f1:.4f}")

with open(os.path.join(MODELS, "baseline.pkl"), "wb") as f:
    pickle.dump({"vectorizer": vec, "classifier": clf}, f)


## 11. Test-Set Evaluation

In [ ]:
# Reload best checkpoints to be sure we are evaluating the best model
bilstm.load_state_dict(torch.load(os.path.join(MODELS, "bilstm_best.pt"), map_location=DEVICE))
hmgs.load_state_dict(torch.load(os.path.join(MODELS, "hmgs_best.pt"), map_location=DEVICE))

bilstm_test = evaluate(bilstm, dl_te, DEVICE, hmgs=False)
hmgs_test = evaluate(hmgs, dl_h_te, DEVICE, hmgs=True)

results = {
    "TF-IDF + LR":      {"accuracy": baseline_test_acc, "macro_f1": baseline_test_f1},
    "BERT-BiLSTM-Attn": {"accuracy": bilstm_test["accuracy"], "macro_f1": bilstm_test["macro_f1"]},
    "HMGS (proposed)":  {"accuracy": hmgs_test["accuracy"], "macro_f1": hmgs_test["macro_f1"]},
}
df = pd.DataFrame(results).T
print(df.to_string(float_format=lambda v: f"{v:.4f}"))


In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
data = [
    ("TF-IDF + LR", baseline_test_preds, lbl_te),
    ("BERT-BiLSTM-Attn", bilstm_test["predictions"], bilstm_test["labels"]),
    ("HMGS (proposed)", hmgs_test["predictions"], hmgs_test["labels"]),
]
for ax, (name, preds, true) in zip(axes, data):
    cm = confusion_matrix(true, preds, labels=[0, 1, 2])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS, ax=ax, cbar=False)
    ax.set_title(name)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "04_confusion_matrices.png"))
plt.show()

reports = {}
for name, preds in [
    ("TF-IDF + LR", baseline_test_preds),
    ("BERT-BiLSTM-Attn", bilstm_test["predictions"]),
    ("HMGS", hmgs_test["predictions"]),
]:
    rep = classification_report(lbl_te, preds, target_names=LABELS,
                                zero_division=0, digits=4)
    reports[name] = rep
    print(f"\n=== {name} ===\n{rep}")
with open(os.path.join(ARTIFACTS, "classification_reports.txt"), "w") as f:
    for name, rep in reports.items():
        f.write(f"=== {name} ===\n{rep}\n\n")


In [ ]:
# Comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
accs = [results[n]["accuracy"] for n in names]
f1s = [results[n]["macro_f1"] for n in names]
x = np.arange(len(names))
w = 0.35
ax.bar(x - w / 2, accs, w, label="Accuracy", color="#10b981")
ax.bar(x + w / 2, f1s, w, label="Macro F1", color="#f59e0b")
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0, 1); ax.set_ylabel("Score")
ax.set_title("Test-Set Performance Comparison")
ax.legend()
for i, (a, b) in enumerate(zip(accs, f1s)):
    ax.text(i - w / 2, a + 0.01, f"{a:.3f}", ha="center", fontsize=9)
    ax.text(i + w / 2, b + 0.01, f"{b:.3f}", ha="center", fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES, "05_model_comparison.png"))
plt.show()


## 12. Interactive Inference

`analyze_text(text)` runs the input through all three models, prints
probability distributions, sentence-level attention from HMGS, and saves a
summary figure to `./artifacts/figures/`. Use it as a question-and-answer
interface: replace the `query` string in the next cell and re-run to get a
prediction for any review.

In [ ]:
@torch.no_grad()
def analyze_text(text, save_figure=True, figure_name="prediction.png"):
    print(f"\n{'=' * 78}")
    print(f"INPUT: {text}")
    print('=' * 78)

    # 1) Baseline
    base_probs = clf.predict_proba(vec.transform([text]))[0]
    base_pred = LABELS[int(base_probs.argmax())]

    # 2) BiLSTM
    bilstm.eval()
    enc = tokenizer(text, max_length=MAX_LEN * 2, padding="max_length",
                    truncation=True, return_tensors="pt")
    enc = {k: v.to(DEVICE) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
    out_b = bilstm(enc["input_ids"], enc["attention_mask"])
    bil_probs = F.softmax(out_b["logits"][0], dim=-1).cpu().numpy()
    bil_pred = LABELS[int(bil_probs.argmax())]

    # 3) HMGS
    hmgs.eval()
    sents = nltk.sent_tokenize(text)[:MAX_SENTS]
    if not sents:
        sents = [text[:200]]
    n = len(sents)
    encs = [tokenizer(s, max_length=MAX_LEN, padding="max_length",
                      truncation=True, return_tensors="pt") for s in sents]
    ids = [e["input_ids"].squeeze(0) for e in encs]
    masks = [e["attention_mask"].squeeze(0) for e in encs]
    while len(ids) < MAX_SENTS:
        ids.append(torch.zeros(MAX_LEN, dtype=torch.long))
        masks.append(torch.zeros(MAX_LEN, dtype=torch.long))
    ids_t = torch.stack(ids, 0).unsqueeze(0).to(DEVICE)
    mask_t = torch.stack(masks, 0).unsqueeze(0).to(DEVICE)
    sent_mask_t = torch.zeros(1, MAX_SENTS, dtype=torch.long, device=DEVICE)
    sent_mask_t[0, :n] = 1
    out_h = hmgs(ids_t, mask_t, sent_mask_t)
    hmgs_probs = F.softmax(out_h["logits"][0], dim=-1).cpu().numpy()
    hmgs_pred = LABELS[int(hmgs_probs.argmax())]
    sent_attn = out_h["attention"][0, :n].cpu().numpy()

    print(f"  TF-IDF + LR        : {base_pred:8s}  "
          f"Neg={base_probs[0]:.3f}  Neu={base_probs[1]:.3f}  Pos={base_probs[2]:.3f}")
    print(f"  BERT-BiLSTM-Attn   : {bil_pred:8s}  "
          f"Neg={bil_probs[0]:.3f}  Neu={bil_probs[1]:.3f}  Pos={bil_probs[2]:.3f}")
    print(f"  HMGS (proposed)    : {hmgs_pred:8s}  "
          f"Neg={hmgs_probs[0]:.3f}  Neu={hmgs_probs[1]:.3f}  Pos={hmgs_probs[2]:.3f}")

    print(f"\nSentence-level breakdown ({n} sentences):")
    for i, s in enumerate(sents):
        print(f"  S{i+1}  attention={sent_attn[i]:.3f}  -> {s[:120]}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    bar_x = np.arange(3); w = 0.25
    axes[0].bar(bar_x - w, base_probs, w, label="TF-IDF + LR", color="#94a3b8")
    axes[0].bar(bar_x, bil_probs, w, label="BiLSTM", color="#6366f1")
    axes[0].bar(bar_x + w, hmgs_probs, w, label="HMGS", color="#10b981")
    axes[0].set_xticks(bar_x); axes[0].set_xticklabels(LABELS)
    axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("Probability")
    axes[0].set_title("Per-model probability distribution")
    axes[0].legend()

    axes[1].barh(np.arange(n), sent_attn, color="#f59e0b")
    axes[1].set_yticks(np.arange(n))
    axes[1].set_yticklabels([f"S{i+1}: {s[:40]}" for i, s in enumerate(sents)])
    axes[1].invert_yaxis()
    axes[1].set_xlabel("Attention Weight")
    axes[1].set_title("HMGS sentence attention")

    short = text[:80] + ("..." if len(text) > 80 else "")
    fig.suptitle(f'Prediction breakdown for: "{short}"', fontsize=10)
    fig.tight_layout()
    if save_figure:
        fig.savefig(os.path.join(FIGURES, figure_name))
    plt.show()

    return {
        "input": text,
        "baseline": {"prediction": base_pred, "probs": base_probs.tolist()},
        "bilstm":   {"prediction": bil_pred, "probs": bil_probs.tolist()},
        "hmgs":     {"prediction": hmgs_pred, "probs": hmgs_probs.tolist(),
                     "sentence_attention": sent_attn.tolist(),
                     "sentences": sents},
    }


In [ ]:
# Demo predictions on canned reviews
demo_inputs = [
    "Best purchase I have made this year. Battery life is amazing and the screen is gorgeous.",
    "Worst product ever. It stopped working after two days and the support team was useless.",
    "It is okay. The build feels cheap but it does the job for the price.",
    "I love the camera but the battery drains too fast.",
]
demo_results = []
for i, text in enumerate(demo_inputs):
    demo_results.append(analyze_text(text, figure_name=f"06_demo_{i+1}.png"))


### Ask Your Own Question
Edit the `query` string below and re-run the cell. The output prints the
prediction and saves a per-query figure to `artifacts/figures/user_query.png`.

In [ ]:
query = "Battery dies in three hours but the screen is gorgeous and the software is fast."
user_result = analyze_text(query, figure_name="07_user_query.png")


## 13. Save Final Summary

In [ ]:
summary = {
    "device": str(DEVICE),
    "in_colab": IN_COLAB,
    "hf_dataset": f"{HF_DATASET} / {HF_CONFIG}",
    "samples": {
        "train": len(texts_tr),
        "val": len(texts_va),
        "test": len(texts_te),
    },
    "hyperparameters": {
        "max_len": MAX_LEN,
        "max_sents": MAX_SENTS,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr_bert": LR_BERT,
        "lr_head": LR_HEAD,
    },
    "test_results": results,
    "bilstm_history": bilstm_history,
    "hmgs_history": hmgs_history,
}
with open(os.path.join(ARTIFACTS, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f"\nAll artifacts saved under: {os.path.abspath(ARTIFACTS)}")
print(f"Figures: {sorted(os.listdir(FIGURES))}")
print(f"Models:  {sorted(os.listdir(MODELS))}")
